[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/15_samplers_and_solvers.ipynb)

# 15. Samplers and numerical solvers

같은 analytic vector field를 여러 적분법으로 한 step씩 진행해 tensor trajectory와 kernel 차이를 비교한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Field and exact reference

dx/dt=-x는 exact solution을 알고 있어 solver 오차를 확인하기 쉽다.


In [ ]:
def f(x, t):
    return -x

x0 = torch.tensor([2.0], device=device)
dt = 0.25
exact = x0 * math.exp(-dt)

print("exact after one step:", exact.item())


## 2. Euler

시작점 slope 하나만 쓴다.


In [ ]:
euler = x0 + dt * f(x0, 0.0)
print("Euler:", euler.item(), "error:", abs(euler - exact).item())


In [ ]:
_ = profile_call("Euler step", lambda x: x + dt * f(x, 0.0), x0)


## 3. Midpoint

중간점 slope를 사용한다.


In [ ]:
k1 = f(x0, 0.0)
mid = x0 + 0.5 * dt * k1
midpoint = x0 + dt * f(mid, dt / 2)

print("Midpoint:", midpoint.item(), "error:", abs(midpoint - exact).item())


In [ ]:
_ = profile_call("Midpoint step", lambda x: x + dt * f(x + 0.5*dt*f(x, 0.0), dt/2), x0)


## 4. Heun

시작과 예측 endpoint slope를 평균한다.


In [ ]:
k1 = f(x0, 0.0)
predict = x0 + dt * k1
k2 = f(predict, dt)
heun = x0 + 0.5 * dt * (k1 + k2)

print("Heun:", heun.item(), "error:", abs(heun - exact).item())


In [ ]:
_ = profile_call("Heun step", lambda x: x + 0.5*dt*(f(x,0.0)+f(x+dt*f(x,0.0),dt)), x0)


## 5. RK4

네 번의 field evaluation을 조합한다.


In [ ]:
k1 = f(x0, 0.0)
k2 = f(x0 + dt*k1/2, dt/2)
k3 = f(x0 + dt*k2/2, dt/2)
k4 = f(x0 + dt*k3, dt)
rk4 = x0 + dt * (k1 + 2*k2 + 2*k3 + k4) / 6

print("RK4:", rk4.item(), "error:", abs(rk4 - exact).item())


In [ ]:
def rk4_step(x):
    k1_ = f(x, 0.0)
    k2_ = f(x + dt * k1_ / 2, dt / 2)
    k3_ = f(x + dt * k2_ / 2, dt / 2)
    k4_ = f(x + dt * k3_, dt)
    return x + dt * (k1_ + 2 * k2_ + 2 * k3_ + k4_) / 6

_ = profile_call("RK4 step", rk4_step, x0)


## 6. DDIM-style deterministic update

predicted x0와 epsilon에서 다음 noise level의 sample을 재구성한다.


In [ ]:
xt = torch.tensor([[1.2, -0.7]], device=device)
pred_x0 = torch.tensor([[1.0, -1.0]], device=device)
pred_eps = torch.tensor([[0.4, 0.6]], device=device)
alpha_next = torch.tensor(0.9, device=device)
sigma_next = torch.sqrt(1 - alpha_next**2)

x_next = alpha_next * pred_x0 + sigma_next * pred_eps
print(x_next)


In [ ]:
_ = profile_call("DDIM-style reconstruction", lambda: alpha_next*pred_x0 + sigma_next*pred_eps)


## 7. Two-step linear multistep

이전 field 값을 재사용하는 LMS 핵심을 본다.


In [ ]:
x = torch.tensor([2.0], device=device)
f_prev = torch.tensor([-2.2], device=device)
f_now = f(x, 0.0)

x_next = x + dt * (1.5 * f_now - 0.5 * f_prev)
print("2-step Adams-Bashforth:", x_next)


In [ ]:
_ = profile_call("2-step LMS", lambda: x + dt*(1.5*f_now - 0.5*f_prev))


## References and provenance

**[15.1] Euler / Heun / RK methods**
- 출처: standard numerical ODE methods
- 이 노트북에서 가져온 부분: flow sampler baselines

**[15.2] DDIM**
- 출처: Song et al., Denoising Diffusion Implicit Models
- 이 노트북에서 가져온 부분: deterministic diffusion update

**[15.3] DPM-Solver family**
- 출처: Lu et al., DPM-Solver / DPM-Solver++
- 이 노트북에서 가져온 부분: high-order diffusion ODE solvers; notebook keeps the core multistep idea small

**[15.4] UniPC / LMS lineages**
- 출처: predictor-corrector and linear multistep diffusion solvers
- 이 노트북에서 가져온 부분: reuse of previous model evaluations
